# 01 Data: cohort construction and record accounting (Table S1)

In [ ]:
import pandas as pd
import numpy as np
import utils

## Cleaning rules, applied the same way to every source

In [ ]:
# columns kept in every built cohort
features = utils.FEATURES              # the eight predictors
subgroup_vars = utils.SUBGROUP_VARS   # race and education, kept for the fairness analysis only
keep_columns = features + subgroup_vars + ["OBITO"]

# codes the registries use to mark an unknown or not-reported value (mapped to missing)
unknown_codes = [("PARTO", 9), ("IDANOMAL", 9), ("APGAR5", 99), ("APGAR1", 99),
                 ("LOCNASC", 9), ("LOCNASC", 4), ("IDADEMAE", 99),
                 ("ESCMAE2010", 9), ("RACACORMAE", 9)]

# valid ranges; values outside become missing (not deleted)
peso_min, peso_max = 250, 5500            # birth weight (grams)
semagestac_min, semagestac_max = 15, 45   # gestational age (weeks)

# neonatal window: a SIM age-at-death code up to 228 means death within 28 days
neonatal_max_idade = 228

# municipality-of-birth code that matches no municipality (one live birth in the national file)
unrecognized_municipality = 520000

flow = []   # rows of Table S1: cohort, step, live births, neonatal deaths
unknown_codes

## São Paulo city birth records, municipal linkage (base_dn_do.csv): training 2012-2014, validation 2015, CEInfo test 2016

In [ ]:
sp = pd.read_csv(utils.RAW + "/base_dn_do.csv", sep=";",
                 usecols=["NUMERODO", "IDADE", "CODMUNNASC", "ANONASC", "GRAVIDEZ"]
                 + features + subgroup_vars)
sp.shape

In [ ]:
# a linked death certificate (NUMERODO present) means the newborn died
sp["OBITO"] = (~sp["NUMERODO"].isna()).astype(int)
sp["OBITO"].value_counts(dropna=False)

In [ ]:
# the development cohort (2012-2015) and the CEInfo 2016 test cohort come from the same source
sp_cohorts = {"São Paulo 2012-2015 (development)": (2012, 2015), "São Paulo 2016 (CEInfo)": (2016, 2016)}
for cohort in sp_cohorts:
    flow.append({"Cohort": cohort, "Step": "São Paulo municipal linkage (CEInfo), 2012-2016 extract",
                 "N": len(sp), "Deaths": int(sp.OBITO.sum())})

sp = sp[sp["CODMUNNASC"] == utils.SP_CODE]
for cohort in sp_cohorts:
    flow.append({"Cohort": cohort, "Step": "São Paulo city (CODMUNNASC == 355030)",
                 "N": len(sp), "Deaths": int(sp.OBITO.sum())})

sp = sp[sp["ANONASC"].between(2012, 2016)]
for cohort, (first_year, last_year) in sp_cohorts.items():
    part = sp[sp["ANONASC"].between(first_year, last_year)]
    flow.append({"Cohort": cohort, "Step": "Births " + str(first_year) + "-" + str(last_year),
                 "N": len(part), "Deaths": int(part.OBITO.sum())})
sp.shape

In [ ]:
# neonatal window: keep survivors and neonatal deaths; drop post-neonatal deaths (IDADE > 228)
sp = sp[(sp["IDADE"] <= neonatal_max_idade) | (sp["IDADE"].isna())]
for cohort, (first_year, last_year) in sp_cohorts.items():
    part = sp[sp["ANONASC"].between(first_year, last_year)]
    flow.append({"Cohort": cohort, "Step": "Neonatal window (drop post-neonatal deaths)",
                 "N": len(part), "Deaths": int(part.OBITO.sum())})
sp.shape

In [ ]:
# singleton pregnancies only
sp = sp[sp["GRAVIDEZ"] == 1]
for cohort, (first_year, last_year) in sp_cohorts.items():
    part = sp[sp["ANONASC"].between(first_year, last_year)]
    flow.append({"Cohort": cohort, "Step": "Single pregnancies only (GRAVIDEZ == 1)",
                 "N": len(part), "Deaths": int(part.OBITO.sum())})
sp.shape

In [ ]:
# when each neonatal death happened (survivors get no timing)
sp["death_timing"] = np.select(
    [sp["IDADE"] < 201, sp["IDADE"].between(201, 207), sp["IDADE"].between(208, 228)],
    ["<1 day", "1-7 days", "8-28 days"], default=None)
sp["death_timing"].value_counts(dropna=False)

In [ ]:
# unknown / not-reported codes to missing
for column, code_value in unknown_codes:
    sp[column] = sp[column].replace(code_value, np.nan)
sp[features + subgroup_vars].isna().mean().round(3)

In [ ]:
# birth weight and gestational age outside the valid range to missing (not deleted)
sp.loc[(sp["PESO"] < peso_min) | (sp["PESO"] > peso_max), "PESO"] = np.nan
sp.loc[(sp["SEMAGESTAC"] < semagestac_min) | (sp["SEMAGESTAC"] > semagestac_max), "SEMAGESTAC"] = np.nan
sp[["PESO", "SEMAGESTAC"]].describe()

In [ ]:
sp = sp[keep_columns + ["ANONASC", "death_timing"]].reset_index(drop=True)
sp.head()

In [ ]:
# split by year into the three cohorts
train = sp[sp["ANONASC"].between(2012, 2014)].reset_index(drop=True)
validation = sp[sp["ANONASC"] == 2015].reset_index(drop=True)
test_ceinfo = sp[sp["ANONASC"] == 2016].reset_index(drop=True)
flow.append({"Cohort": "São Paulo 2012-2014 (training)", "Step": "Analyzed set",
             "N": len(train), "Deaths": int(train.OBITO.sum())})
flow.append({"Cohort": "São Paulo 2015 (validation)", "Step": "Analyzed set",
             "N": len(validation), "Deaths": int(validation.OBITO.sum())})
pd.DataFrame({
    "cohort": ["train_2012_2014", "validation_2015", "test_ceinfo_2016"],
    "n": [len(train), len(validation), len(test_ceinfo)],
    "deaths": [int(train.OBITO.sum()), int(validation.OBITO.sum()), int(test_ceinfo.OBITO.sum())]})

## National birth records, Ministry of Health linkage, Brazil 2016 (DNBRLK16.DBF): external test and São Paulo local test

In [ ]:
national = utils.read_dbf(utils.RAW + "/DNBRLK16.DBF",
    ["CODMUNNASC", "LOCNASC", "IDADEMAE", "APGAR1", "APGAR5", "PESO", "SEMAGESTAC",
     "IDANOMAL", "PARTO", "ESCMAE2010", "RACACOR", "GRAVIDEZ", "NUMERODO", "IDADEDO"])
national.shape

In [ ]:
# read_dbf returns text columns: define the outcome, convert the numeric columns, align the column names
national["OBITO"] = (national["NUMERODO"].str.len() > 0).astype(int)
for column in ["CODMUNNASC", "LOCNASC", "IDADEMAE", "APGAR1", "APGAR5", "PESO", "SEMAGESTAC",
               "IDANOMAL", "PARTO", "ESCMAE2010", "RACACOR", "GRAVIDEZ", "IDADEDO"]:
    national[column] = pd.to_numeric(national[column].replace("", np.nan), errors="coerce")
national = national.rename(columns={"RACACOR": "RACACORMAE", "IDADEDO": "IDADE"})
flow.append({"Cohort": "Brazil 2016 (national linkage)", "Step": "SINASC-SIM linkage (all Brazil)",
             "N": len(national), "Deaths": int(national.OBITO.sum())})
national["OBITO"].value_counts(dropna=False)

In [ ]:
# same recipe: neonatal window, then singleton pregnancies, then the unrecognized municipality code
national = national[(national["IDADE"] <= neonatal_max_idade) | (national["IDADE"].isna())]
flow.append({"Cohort": "Brazil 2016 (national linkage)", "Step": "Neonatal window (drop post-neonatal deaths)",
             "N": len(national), "Deaths": int(national.OBITO.sum())})
national = national[national["GRAVIDEZ"] == 1]
flow.append({"Cohort": "Brazil 2016 (national linkage)", "Step": "Single pregnancies only (GRAVIDEZ == 1)",
             "N": len(national), "Deaths": int(national.OBITO.sum())})
national = national[national["CODMUNNASC"] != unrecognized_municipality]
flow.append({"Cohort": "Brazil 2016 (national linkage)", "Step": "Drop the one birth with an unrecognized municipality code (520000)",
             "N": len(national), "Deaths": int(national.OBITO.sum())})
national.shape

In [ ]:
# death timing, unknown codes to missing, out-of-range to missing (same rules as above)
national["death_timing"] = np.select(
    [national["IDADE"] < 201, national["IDADE"].between(201, 207), national["IDADE"].between(208, 228)],
    ["<1 day", "1-7 days", "8-28 days"], default=None)
for column, code_value in unknown_codes:
    national[column] = national[column].replace(code_value, np.nan)
national.loc[(national["PESO"] < peso_min) | (national["PESO"] > peso_max), "PESO"] = np.nan
national.loc[(national["SEMAGESTAC"] < semagestac_min)
             | (national["SEMAGESTAC"] > semagestac_max), "SEMAGESTAC"] = np.nan
national = national[keep_columns + ["CODMUNNASC", "death_timing"]].reset_index(drop=True)
national.head()

In [ ]:
# the external test set keeps every municipality (São Paulo city included as one municipality);
# the local test set is its São Paulo city subset
test_external = national.reset_index(drop=True)
test_local = national[national["CODMUNNASC"] == utils.SP_CODE].reset_index(drop=True)
other_cities = national[national["CODMUNNASC"] != utils.SP_CODE]
flow.append({"Cohort": "São Paulo 2016 local (MoH)", "Step": "Analyzed set, filter São Paulo city",
             "N": len(test_local), "Deaths": int(test_local.OBITO.sum())})
flow.append({"Cohort": "Brazil 2016 (external)", "Step": "Analyzed set, all municipalities except São Paulo city",
             "N": len(other_cities), "Deaths": int(other_cities.OBITO.sum())})
pd.DataFrame({
    "cohort": ["test_external_brazil_2016", "test_local_2016_moh"],
    "n": [len(test_external), len(test_local)],
    "deaths": [int(test_external.OBITO.sum()), int(test_local.OBITO.sum())]})

## São Paulo city temporal records, 2017-2020 (DNSP_DOI_2012_2020.csv)

In [ ]:
# this file uses comma decimals (decimal=",")
temporal = pd.read_csv(utils.RAW + "/DNSP_DOI_2012_2020.csv", sep=";", decimal=",",
                       encoding="latin-1", on_bad_lines="skip",
                       usecols=["CODMUNNASC", "ANONASC", "TPOBITO", "IDADE", "GRAVIDEZ"]
                       + features + subgroup_vars)
for column in ["CODMUNNASC", "ANONASC", "TPOBITO", "IDADE", "GRAVIDEZ"] + features + subgroup_vars:
    temporal[column] = pd.to_numeric(temporal[column], errors="coerce")
temporal.shape

In [ ]:
# in this file a neonatal death is coded as TPOBITO == 2
temporal["OBITO"] = temporal["TPOBITO"].eq(2).astype(int)
flow.append({"Cohort": "São Paulo 2017-2020 (temporal)", "Step": "São Paulo temporal linkage",
             "N": len(temporal), "Deaths": int(temporal.OBITO.sum())})
temporal["OBITO"].value_counts(dropna=False)

In [ ]:
temporal = temporal[temporal["CODMUNNASC"] == utils.SP_CODE]
flow.append({"Cohort": "São Paulo 2017-2020 (temporal)", "Step": "São Paulo city (CODMUNNASC == 355030)",
             "N": len(temporal), "Deaths": int(temporal.OBITO.sum())})
temporal = temporal[temporal["ANONASC"].between(2017, 2020)]
flow.append({"Cohort": "São Paulo 2017-2020 (temporal)", "Step": "Births 2017-2020",
             "N": len(temporal), "Deaths": int(temporal.OBITO.sum())})
temporal["ANONASC"].value_counts().sort_index()

In [ ]:
# same recipe: neonatal window, singleton pregnancies
temporal = temporal[(temporal["IDADE"] <= neonatal_max_idade) | (temporal["IDADE"].isna())]
flow.append({"Cohort": "São Paulo 2017-2020 (temporal)", "Step": "Neonatal window (drop post-neonatal deaths)",
             "N": len(temporal), "Deaths": int(temporal.OBITO.sum())})
temporal = temporal[temporal["GRAVIDEZ"] == 1]
flow.append({"Cohort": "São Paulo 2017-2020 (temporal)", "Step": "Single pregnancies only (GRAVIDEZ == 1)",
             "N": len(temporal), "Deaths": int(temporal.OBITO.sum())})
temporal.shape

In [ ]:
# death timing, unknown codes to missing, out-of-range to missing (same rules)
temporal["death_timing"] = np.select(
    [temporal["IDADE"] < 201, temporal["IDADE"].between(201, 207), temporal["IDADE"].between(208, 228)],
    ["<1 day", "1-7 days", "8-28 days"], default=None)
for column, code_value in unknown_codes:
    temporal[column] = temporal[column].replace(code_value, np.nan)
temporal.loc[(temporal["PESO"] < peso_min) | (temporal["PESO"] > peso_max), "PESO"] = np.nan
temporal.loc[(temporal["SEMAGESTAC"] < semagestac_min)
             | (temporal["SEMAGESTAC"] > semagestac_max), "SEMAGESTAC"] = np.nan
temporal = temporal[keep_columns + ["ANONASC", "death_timing"]].reset_index(drop=True)
temporal.head()

## Save the six cohorts

In [ ]:
cohorts = {
    "train_2012_2014": train,
    "validation_2015": validation,
    "test_local_2016_moh": test_local,
    "test_ceinfo_2016": test_ceinfo,
    "test_external_brazil_2016": test_external,
    "test_temporal_sp_2017_2020": temporal,
}
saved = []
for name, cohort in cohorts.items():
    utils.save_object(cohort, utils.DATA + "/" + name)
    saved.append({"cohort": name, "n": len(cohort), "deaths": int(cohort.OBITO.sum()),
                  "prevalence_%": round(100 * cohort.OBITO.mean(), 3)})
pd.DataFrame(saved)

## Table S1: record accounting from the source files to the analyzed sets

In [ ]:
cohort_order = ["São Paulo 2012-2015 (development)", "São Paulo 2012-2014 (training)",
                "São Paulo 2015 (validation)", "Brazil 2016 (national linkage)",
                "São Paulo 2016 local (MoH)", "Brazil 2016 (external)",
                "São Paulo 2016 (CEInfo)", "São Paulo 2017-2020 (temporal)"]
cohort_flow = pd.DataFrame(flow)
cohort_flow["order"] = cohort_flow["Cohort"].map({c: i for i, c in enumerate(cohort_order)})
cohort_flow = cohort_flow.sort_values("order", kind="stable").drop(columns="order").reset_index(drop=True)
utils.save_table(cohort_flow, utils.SUPPLEMENT_TABLES, "Table_S1")
cohort_flow